# Predictive Defect Rate — 00 Data Quality

**Source**: `Predictive_DefectRate_Plan.md` Phase B §3.3 데이터 품질 사전 점검 체크리스트.

Run after `tools/export_training_data.py` produces a parquet file. This notebook is the GATE between Phase B and Phase C — if any checklist item fails red, do NOT proceed to baseline training.

## Checklist
- [ ] 최소 3개월치 InspectionHistory 누적 (6개월 권장)
- [ ] NG율 > 0.2%, < 50% (한쪽 극단치 모이면 회귀 부적합)
- [ ] 컬럼별 NULL 비율 확인 (V1/V2/V3 피처 가용성)
- [ ] DefectCode 라벨링 일관성 (Web UI 자동완성 도입됨 — 별도 점검)
- [ ] 시계열 갭 점검 (장시간 무가동 구간)
- [ ] (Client, Recipe) 페어별 데이터 양 분포 (불균형 시 트레이닝 split 전략 변경)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

PARQUET_PATH = Path('../data/training_v1.parquet')  # adjust if needed
assert PARQUET_PATH.exists(), f'Run export_training_data.py first → {PARQUET_PATH}'

df = pd.read_parquet(PARQUET_PATH)
print(f'rows: {len(df):,}  cols: {len(df.columns)}')
df.head()

## 1. 데이터 양 — 누적 기간 / Client·Recipe 페어

Plan §3.3 요구: 최소 3개월, 권장 6개월. 또한 페어별 최소 buckets 확인.

In [ ]:
df['HourBucket'] = pd.to_datetime(df['HourBucket'])
span_days = (df['HourBucket'].max() - df['HourBucket'].min()).days
print(f'기간       : {df["HourBucket"].min()} → {df["HourBucket"].max()}  ({span_days} days)')
print(f'고유 client: {df["ClientId"].nunique()}')
print(f'고유 recipe: {df["RecipeName"].nunique()}')

pair_counts = df.groupby(['ClientId', 'RecipeName']).size().sort_values(ascending=False)
print('\n페어별 buckets (top/bottom 5):')
print(pair_counts.head())
print(pair_counts.tail())

## 2. 타깃 NG율 분포

극단(전부 OK / 전부 NG)에 몰리면 회귀가 무의미.

In [ ]:
tgt = df['TargetNgRate']
print(tgt.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99]))
print(f'\nzero-NG 비율  : {(tgt == 0).mean():.2%}')
print(f'전체NG 비율   : {(tgt == 1).mean():.2%}')
print(f'plan 임계 0.2% 미만 비율 : {(tgt < 0.002).mean():.2%}')
print(f'plan 임계 50%   초과 비율: {(tgt > 0.5).mean():.2%}')

## 3. 컬럼별 NULL 비율 — V1/V2/V3 피처 가용성

VMS 미업데이트 클라이언트가 섞여 있으면 신규 피처가 모두 NULL. 100% NULL 인 피처는 모델에 무의미.

In [ ]:
null_pct = (df.isna().mean() * 100).sort_values(ascending=False)
null_pct[null_pct > 0]

## 4. 시계열 갭 — 페어별 연속성

Bucket 간격이 1h 가 아닌 곳 = 무가동 시간. 너무 많으면 LSTM/시계열 모델은 reindex 필요.

In [ ]:
gaps = (
    df.sort_values(['ClientId', 'RecipeName', 'HourBucket'])
      .groupby(['ClientId', 'RecipeName'])['HourBucket']
      .diff()
      .dropna()
)
gap_hours = gaps.dt.total_seconds() / 3600
print('Bucket 간격(시간):')
print(gap_hours.describe(percentiles=[0.5, 0.9, 0.99]))
print(f'\n>1h 갭 비율 : {(gap_hours > 1.001).mean():.2%}')
print(f'>24h 갭 비율: {(gap_hours > 24).mean():.2%}')

## 5. 컨텍스트별 NG율 — 신호 vs 노이즈 사전 점검

ShiftId / 작업자 수 / DL Confidence 와 타깃 사이에 의미 있는 차이가 보이면 베이스라인이 잡힐 가능성 높음.

In [ ]:
by_shift = df.groupby('ShiftId')['TargetNgRate'].agg(['count', 'mean', 'std'])
print('Shift 별 타깃 NG율:')
print(by_shift)

# DL confidence 가 있는 행만 — quartile 별 NG율
with_dl = df.dropna(subset=['MinDlConfidence']).copy()
if len(with_dl) > 0:
    with_dl['conf_bin'] = pd.qcut(with_dl['MinDlConfidence'], q=4, duplicates='drop')
    print('\nDL Confidence quartile 별 타깃 NG율:')
    print(with_dl.groupby('conf_bin', observed=True)['TargetNgRate'].agg(['count', 'mean']))

## 6. 게이트 결정

위 결과를 보고 다음 중 하나:

1. **PASS** → Phase C LightGBM baseline 진행
2. **FAIL 데이터 부족** → 데이터 더 누적될 때까지 대기 (VMS 운영 지속)
3. **FAIL 라벨 불균형** → 회귀 대신 binary NG 발생 여부 분류로 문제 재정의
4. **FAIL 피처 누락** → V1/V2/V3 미배포 VMS 클라이언트 식별 후 업데이트